<a href="https://colab.research.google.com/github/neslihanyildizozhan/ABCD-COI/blob/main/1_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

abcd_p_demo = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/abcd-general/abcd_p_demo.csv", low_memory=False)
ph_y_anthro = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/physical-health/ph_y_anthro.csv", low_memory=False)
abcd_y_lt = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/abcd-general/abcd_y_lt.csv",low_memory=False)
mh_p_cbcl = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/mental-health/mh_p_cbcl.csv",low_memory=False)
ph_y_pds = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/physical-health/ph_y_pds.csv", low_memory=False)
led_l_coi = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/linked-external-data/led_l_coi.csv", low_memory=False)
mh_y_pps = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/mental-health/mh_y_pps.csv",low_memory=False)
ph_p_sds = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/physical-health/ph_p_sds.csv",low_memory=False)
nc_y_nih = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/neurocognition/nc_y_nihtb.csv",low_memory=False)
abcd_y_lt = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/abcd-general/abcd_y_lt.csv",low_memory=False)
abcd_y_lt = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/abcd-general/abcd_y_lt.csv",low_memory=False)
mri_y_qc_motion = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/imaging/mri_y_qc_motion.csv",low_memory=False)
mh_p_fhx = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/mental-health/mh_p_fhx.csv",low_memory=False)
dmri_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/ABCDCorrected_Jan2024/ABCD-DiffusionMeasure-data_NDA.csv",low_memory=False)


# MAIN MODEL for NIH
-covariables: age,sex, race, ethnicity, bmi, totalincome, parenteducation, sleep, puberty, site, family id


In [ ]:
coi_levels = led_l_coi[['src_subject_id', 'eventname', 'reshist_addr1_coi_r_coi_nat','reshist_addr1_coi_r_se_nat','reshist_addr1_coi_r_he_nat','reshist_addr1_coi_r_ed_nat']]
coi_levels.rename(columns={'reshist_addr1_coi_r_coi_nat': 'coi_total_raw', 'reshist_addr1_coi_r_he_nat': 'coi_he_raw',
                                 'reshist_addr1_coi_r_se_nat':'coi_se_raw','reshist_addr1_coi_r_ed_nat' : 'coi_ed_raw'
                                 }, inplace=True)
print(coi_levels)

In [ ]:
dmri_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/ABCDCorrected_Jan2024/ABCD-DiffusionMeasure-data_NDA.csv",low_memory=False)

dmri_data.rename(columns=dmri_data.iloc[0], inplace=True)
dmri_data.drop([0], inplace=True)

ten1_fa_cols = ['src_subject_id',
    'AF_left_Ten1_FA', 'AF_right_Ten1_FA',
    'CB_left_Ten1_FA', 'CB_right_Ten1_FA',
    'EmC_left_Ten1_FA', 'EmC_right_Ten1_FA',
    'ILF_left_Ten1_FA', 'ILF_right_Ten1_FA',
    'IOFF_left_Ten1_FA', 'IOFF_right_Ten1_FA',
    'MdLF_left_Ten1_FA', 'MdLF_right_Ten1_FA',
    'SLF_I_left_Ten1_FA', 'SLF_I_right_Ten1_FA',
    'SLF_II_left_Ten1_FA', 'SLF_II_right_Ten1_FA',
    'SLF_III_left_Ten1_FA', 'SLF_III_right_Ten1_FA',
    'UF_left_Ten1_FA', 'UF_right_Ten1_FA'
]

dmri_ten1_fa = dmri_data[ten1_fa_cols]

coi_levels = pd.merge(coi_levels, dmri_ten1_fa, on ='src_subject_id', how ='left')

for column in coi_levels.columns:
    if column.endswith('_Ten1_FA'):
        coi_levels[column] = coi_levels[column].astype(float)


ten1_fa_cols = [col for col in coi_levels.columns if col.endswith('_Ten1_FA')]

coi_levels = coi_levels[~(coi_levels[ten1_fa_cols] == -1).any(axis=1)]

coi_levels.dropna(inplace=True)



In [ ]:
abcd_p_demo_selected = abcd_p_demo[['src_subject_id','eventname','demo_sex_v2',
                                    'demo_ethn_v2','demo_prnt_ed_v2','demo_prtnr_ed_v2',
                                    'demo_comb_income_v2']]

for col in abcd_p_demo.columns:
  if col.startswith('demo_race_a_p___'):
    abcd_p_demo_selected[col] = abcd_p_demo[col]

abcd_p_demo_selected = abcd_p_demo_selected.query("eventname == 'baseline_year_1_arm_1'")

# Convert the 'demo_sex_v2' column to a Categorical object as male - female

abcd_p_demo_selected['demo_sex_v2'] = pd.Categorical(abcd_p_demo_selected['demo_sex_v2'], categories=[1, 2], ordered=False)
abcd_p_demo_selected['demo_sex_v2'] = abcd_p_demo_selected['demo_sex_v2'].replace({1: 'male', 2: 'female'})


#ethnicity
abcd_p_demo_selected['demo_ethn_v2'] = abcd_p_demo_selected['demo_ethn_v2'].replace({1: 'hispanic', 2: 'non-hispanic'})
#Replace 777 and 999 in ethnicity
abcd_p_demo_selected['demo_ethn_v2'] = abcd_p_demo_selected['demo_ethn_v2'].replace([777, 999], np.nan)



#race
def consolidate_race(row):
    for col in ['demo_race_a_p___10']:
        if row[col] == 1:
            return 'White'
    for col in ['demo_race_a_p___11']:
        if row[col] == 1:
            return 'Black'
    for col in ['demo_race_a_p___18', 'demo_race_a_p___19', 'demo_race_a_p___20', 'demo_race_a_p___21', 'demo_race_a_p___22', 'demo_race_a_p___23', 'demo_race_a_p___24']:
        if row[col] == 1:
            return 'Asian'
    return 'Other'

abcd_p_demo_selected['race'] = abcd_p_demo_selected.apply(consolidate_race, axis=1)


In [ ]:
#In education and household total income data in ABCD Study contains 777 and 999 values for 'do not know' or 'refuse to answer'
# Replace 777 and 999 with NaN in the education columns and omit unknown values

abcd_p_demo_selected['demo_prnt_ed_v2'] = abcd_p_demo_selected['demo_prnt_ed_v2'].replace([777, 999], np.nan)
abcd_p_demo_selected['demo_prtnr_ed_v2'] = abcd_p_demo_selected['demo_prtnr_ed_v2'].replace([777, 999], np.nan)

def calculate_highest_education(row):
    mother = row['demo_prnt_ed_v2']
    father = row['demo_prtnr_ed_v2']
    if pd.isna(mother) and pd.isna(father):
        return np.nan
    elif pd.isna(mother):
        return father
    elif pd.isna(father):
        return mother
    else:
        return max(mother, father)

abcd_p_demo_selected['highest_education'] = abcd_p_demo_selected.apply(calculate_highest_education, axis=1)


# Education and income categorization
#education {range(0, 13): '<high school', range(13, 15): 'High school-GED', range(15, 18): 'College', 18: 'Bachelor', range(19, 22): 'Post-Grad'}
# income = {range(1, 5): '<25k', range(5, 7): '25-49k', 7: '50-74k', 8: '75-99k', 9: '100-199k', 10: '>200k'}

#totalincome variable
#Replace 777 and 999 in income column and categorize it
abcd_p_demo_selected['demo_comb_income_v2'] = abcd_p_demo_selected['demo_comb_income_v2'].replace([777, 999], np.nan)


def categorize_income(income):
    if 1 <= income <= 4:
        return '<25k'
    elif 5 <= income <= 6:
        return '25-49k'
    elif income == 7:
        return '50-74k'
    elif income == 8:
        return '75-99k'
    elif income == 9:
        return '100-199k'
    elif income == 10:
        return '>200k'
    else:
        return np.nan

abcd_p_demo_selected['demo_comb_income_v2'] = abcd_p_demo_selected['demo_comb_income_v2'].apply(categorize_income)


abcd_p_demo_selected['highest_education'] = abcd_p_demo_selected['highest_education'].replace({
    13: '12',
    14: '12',
    15: '14',
    16: '14',
    17: '14',
    18: '16',
    19: '18',
    21: '22'
})


abcd_p_demo_selected['highest_education'] = pd.to_numeric(abcd_p_demo_selected['highest_education'])

abcdDemo = abcd_p_demo_selected[['src_subject_id', 'eventname', 'demo_sex_v2', 'race','demo_ethn_v2',
               'highest_education', 'demo_comb_income_v2']]

abcdDemo.rename(columns={ 'demo_sex_v2': 'sex', 'demo_ethn_v2': 'ethnicity',
                         'highest_education': 'parenteducation', 'demo_comb_income_v2': 'totalincome'}, inplace=True)

In [ ]:
interview_general = abcd_y_lt[['src_subject_id','eventname','interview_age','rel_family_id','site_id_l']]
interview_general = interview_general.query("eventname == 'baseline_year_1_arm_1'")

abcdDemo = pd.merge(interview_general, abcdDemo , on=['src_subject_id', 'eventname'], how='left')
abcd_coi = pd.merge(abcdDemo, coi_levels , on= ['src_subject_id', 'eventname'], how='left')


In [ ]:
# BMI calculation
ph_y_anthro_selected = ph_y_anthro[['src_subject_id','eventname','anthroweightcalc','anthroheightcalc']]

for column in ph_y_anthro_selected.columns[2:]:
  print(f"Column: {column}")
  print(ph_y_anthro_selected[column].value_counts(dropna=False))
  print("\n")

bmi = ph_y_anthro_selected.query("eventname == 'baseline_year_1_arm_1'").dropna()

bmi['bmi'] = (bmi['anthroweightcalc'] / (bmi['anthroheightcalc'] ** 2)) * 703
bmi = bmi[bmi['bmi'] < 1000]

bmi = bmi[['src_subject_id','eventname','bmi']]
abcd_coi = pd.merge(abcd_coi, bmi, on=['src_subject_id', 'eventname'], how='left')
print(abcd_coi)

In [ ]:

coi_cols = [col for col in abcd_coi.columns if col.startswith('coi')]
selected_cols = ['src_subject_id', 'eventname', 'interview_age', 'sex','race','ethnicity',
                 'rel_family_id', 'site_id_l', 'totalincome', 'parenteducation',
                 'bmi'] + coi_cols + ten1_fa_cols


result_df = abcd_coi[selected_cols]


nih = nc_y_nih[['src_subject_id', 'eventname', 'nihtbx_totalcomp_uncorrected','nihtbx_picvocab_uncorrected',
                'nihtbx_flanker_uncorrected', 'nihtbx_list_uncorrected','nihtbx_cardsort_uncorrected','nihtbx_pattern_uncorrected','nihtbx_picture_uncorrected', 'nihtbx_reading_uncorrected',
                'nihtbx_fluidcomp_uncorrected','nihtbx_cryst_uncorrected']]
nih = nih[nih['eventname'] == "baseline_year_1_arm_1"]

for column in nih.columns[2:]:
  print(f"Column: {column}")
  print(nih[column].value_counts(dropna=False))
  print("\n")
result_df = pd.merge(result_df, nih, on=['src_subject_id', 'eventname'], how='left')


for column in result_df.columns[2:]:
  print(f"Column: {column}")
  print(result_df[column].value_counts(dropna=False))
  print("\n")

print(result_df)

In [ ]:
# Puberty stages
puberty_stages_all = ph_y_pds[['src_subject_id', 'eventname','pds_y_ss_female_category_2','pds_y_ss_male_cat_2']]
puberty_stages = puberty_stages_all.query("eventname == 'baseline_year_1_arm_1'")

result_df = pd.merge(result_df, puberty_stages, on=['src_subject_id', 'eventname'], how='left')
result_df['puberty_stage'] = np.where(result_df['sex'] == 'male', result_df['pds_y_ss_male_cat_2'], result_df['pds_y_ss_female_category_2'])

result_df['puberty_stage'] = result_df['puberty_stage'].replace({1: 'prepuberty', 2: 'early puberty', 3: 'mid puberty', 4: 'late/post puberty', 5: 'late/post puberty'})
result_df = result_df.drop(columns=['pds_y_ss_male_cat_2', 'pds_y_ss_female_category_2'])


In [ ]:

# Sleep hours
sleep_hrs_all = ph_p_sds[['src_subject_id', 'eventname','sleepdisturb1_p']]
sleep_hrs = sleep_hrs_all.query("eventname == 'baseline_year_1_arm_1'").dropna()
sleep_hrs['sleep_category'] = np.where(sleep_hrs['sleepdisturb1_p'] == 1, '9 hours + more', 'less than 9 hours')
sleep_hrs = sleep_hrs[['src_subject_id', 'eventname', 'sleep_category']]

result_df = pd.merge(result_df, sleep_hrs, on=['src_subject_id', 'eventname'], how='left')
print(result_df)


In [ ]:
import pandas as pd
import numpy as np
mh_p_fhx = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abcd-data-release-5.1 (1)/core/mental-health/mh_p_fhx.csv",low_memory=False)
fam_h = mh_p_fhx[['src_subject_id','eventname', 'fam_history_q7a_mania', 'fam_history_q7d_mania','fam_history_q6a_depression','fam_history_q6d_depression',
                  'fam_history_q12a_hospitalized','fam_history_q12d_hospitalized','fam_history_q8a_visions','fam_history_q8d_visions',
                  'fam_history_q10d_nerves','fam_history_q10a_nerves']]
fam_h = fam_h.query("eventname == 'baseline_year_1_arm_1'")
fam_h.replace(999, np.nan, inplace=True)

fam_h['momdad_mania'] = fam_h[['fam_history_q7a_mania', 'fam_history_q7d_mania']].max(axis=1)
fam_h['momdad_dep'] = fam_h[['fam_history_q6a_depression', 'fam_history_q6d_depression']].max(axis=1)
fam_h['momdad_hospitalized'] = fam_h[['fam_history_q12a_hospitalized', 'fam_history_q12d_hospitalized']].max(axis=1)
fam_h['momdad_nerves'] = fam_h[['fam_history_q10d_nerves', 'fam_history_q10a_nerves']].max(axis=1)
fam_h['momdad_visions'] = fam_h[['fam_history_q8a_visions', 'fam_history_q8d_visions']].max(axis=1)
fam_h['fam_hx'] = fam_h[['momdad_mania', 'momdad_dep', 'momdad_hospitalized', 'momdad_visions','momdad_nerves']].max(axis=1)
fam_h['fam_hx'] = fam_h['fam_hx'].map({1: 'yes', 0: 'no'})
fam_h['fam_hx'].fillna('no', inplace=True)
fam_h = fam_h[['src_subject_id', 'eventname', 'fam_hx']]

result_df = pd.merge(result_df, fam_h, on=['src_subject_id', 'eventname'], how='left')
print(result_df)

In [ ]:
#head motion
mri_y_qc_motion = mri_y_qc_motion[['src_subject_id', 'eventname','dmri_meanmotion']]
mri_y_qc_motion = mri_y_qc_motion.query("eventname == 'baseline_year_1_arm_1'")

abcd_coi = pd.merge(result_df, mri_y_qc_motion, on=['src_subject_id', 'eventname'], how='left')

print(abcd_coi)


In [ ]:
#exclusion due to missing variables
abcd_coi.dropna(subset=['coi_total_raw'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['interview_age'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['sex'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['race'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['ethnicity'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['totalincome'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['parenteducation'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['bmi'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['puberty_stage'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['sleep_category'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['fam_hx'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['site_id_l'], inplace=True)
print(abcd_coi['src_subject_id'].count())

  #dmri_meanmotion
abcd_coi.dropna(subset=['dmri_meanmotion'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna(subset=['rel_family_id'], inplace=True)
print(abcd_coi['src_subject_id'].count())

abcd_coi.dropna( inplace=True)
print(abcd_coi['src_subject_id'].count())


# Main model for NIH subject: 7895 without nan values

#Demographics

In [ ]:

categorical_vars = ['sex', 'race', 'ethnicity', 'totalincome','puberty_stage','sleep_category','fam_hx']

for var in categorical_vars:
  print(f"Value counts for {var}:\n{abcd_coi[var].value_counts(dropna=False)}\n")
  print(f"Percentages for {var}:\n{abcd_coi[var].value_counts(normalize=True, dropna=False) * 100}\n\n")

continuous_vars = ['interview_age', 'bmi','parenteducation','coi_total_raw','coi_ed_raw','coi_se_raw','coi_he_raw','nihtbx_totalcomp_uncorrected','nihtbx_fluidcomp_uncorrected','nihtbx_cryst_uncorrected']

for var in continuous_vars:
  print(f"Mean of {var}: {abcd_coi[var].mean()}")
  print(f"Standard deviation of {var}: {abcd_coi[var].std()}\n\n")


In [ ]:

for var in ['interview_age','coi_total_raw', 'coi_ed_raw', 'coi_se_raw', 'coi_he_raw', 'bmi', 'parenteducation','nihtbx_totalcomp_uncorrected','nihtbx_fluidcomp_uncorrected','nihtbx_cryst_uncorrected']:
  print(f"Range of {var}: [{abcd_coi[var].min()}, {abcd_coi[var].max()}]")

#Correlation Matrix

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

data = abcd_coi[['coi_total_raw','interview_age','sex','race','ethnicity','totalincome','parenteducation',
                 'bmi','puberty_stage','sleep_category','fam_hx','site_id_l','dmri_meanmotion','rel_family_id']]

data['sex'] = data['sex'].astype('category').cat.codes
data['race'] = data['race'].astype('category').cat.codes
data['ethnicity'] = data['ethnicity'].astype('category').cat.codes
data['totalincome'] = data['totalincome'].astype('category').cat.codes
data['puberty_stage'] = data['puberty_stage'].astype('category').cat.codes
data['sleep_category'] = data['sleep_category'].astype('category').cat.codes
data['fam_hx'] = data['fam_hx'].astype('category').cat.codes
data['site_id_l'] = data['site_id_l'].astype('category').cat.codes


correlation_matrix = data.corr()

print(correlation_matrix)

high_corr_pairs = [(i, j) for i in correlation_matrix.columns for j in correlation_matrix.columns if i != j and abs(correlation_matrix.loc[i, j]) > 0.05]
print(high_corr_pairs)


import matplotlib.pyplot as plt

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

f, ax = plt.subplots(figsize=(11, 9))

cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(correlation_matrix,  annot=True, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})

plt.show()

In [ ]:
#VIF

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


X = data
X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

In [ ]:
abcd_coi.to_csv("abcd_coi.csv")

In [ ]:
#Distribution
plt.figure(figsize=(8, 6))
plt.scatter(abcd_coi['coi_total_raw'], abcd_coi['nihtbx_totalcomp_uncorrected'])
plt.xlabel('coi_total_raw')
plt.ylabel('nihtbx_totalcomp_uncorrected')
plt.show()